# Random Forest Baseline Model

**Module:** IT3051 - Fundamentals of Data Mining  
**Project:** Next-Year PM2.5 Air-Quality Risk Classification  
**Model:** Random Forest Classifier  
**Purpose:** Baseline model development and validation

This notebook:

- Loads the chronological train/validation/test datasets created by `07_make_dataset.ipynb`
- Reuses the preprocessing logic from `10_preprocessing_pipeline.ipynb`
- Fits preprocessing **only on training data** to prevent leakage
- Trains a baseline Random Forest model
- Evaluates the model on the validation set
- Reports Accuracy, Macro F1, Classification Report, and Confusion Matrix
- Keeps the test set untouched for later final-model evaluation

> This is a **baseline** Random Forest. Hyperparameter tuning and model optimization are intentionally not performed here.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# Make this work when the notebook is run from the project's notebooks folder
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing.preprocessing_pipeline import (
    TARGET_COLUMN,
    prepare_base_dataframe,
    prepare_features_and_target,
    build_preprocessor,
)

print("Imports loaded successfully.")
print("Project root:", project_root)


## 1. Load Chronological Split Datasets

`07_make_dataset.ipynb` creates chronological splits:

- Training: years up to 2012
- Validation: 2013-2014
- Test: 2015 onward

We load the full split files because they contain both the predictors and target.


In [ ]:
processed_dir = project_root / "data" / "processed"

train_path = processed_dir / "train_data.csv"
validation_path = processed_dir / "validation_data.csv"
test_path = processed_dir / "test_data.csv"

required_files = [train_path, validation_path, test_path]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}\n"
            "Run 07_make_dataset.ipynb first to create the chronological split files."
        )

identifier_types = {
    "site_id": "string",
    "state_code": "string",
    "county_code": "string",
    "site_num": "string",
}

train_df = pd.read_csv(train_path, dtype=identifier_types, low_memory=False)
validation_df = pd.read_csv(validation_path, dtype=identifier_types, low_memory=False)
test_df = pd.read_csv(test_path, dtype=identifier_types, low_memory=False)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)


## 2. Confirm the Chronological Split

This check helps confirm that future years are not mixed into the training data.


In [ ]:
for name, dataset in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df),
]:
    print(
        f"{name}: {dataset['year'].min()} - {dataset['year'].max()} "
        f"({len(dataset)} rows)"
    )


## 3. Apply Base Cleaning to Each Split

We use the same base-cleaning function used in the preprocessing notebook.  
Each split is handled separately.


In [ ]:
train_prepared = prepare_base_dataframe(train_df)
validation_prepared = prepare_base_dataframe(validation_df)
test_prepared = prepare_base_dataframe(test_df)

print("Base preprocessing completed.")
print("Training shape:", train_prepared.shape)
print("Validation shape:", validation_prepared.shape)
print("Test shape:", test_prepared.shape)


## 4. Separate Features and Target

The target is `pm25_risk_category_next_year`.

Identifier columns are excluded by the existing project preprocessing function.


In [ ]:
X_train, y_train = prepare_features_and_target(train_prepared)
X_val, y_val = prepare_features_and_target(validation_prepared)
X_test, y_test = prepare_features_and_target(test_prepared)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:", X_val.shape, "| y_val:", y_val.shape)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())


## 5. Build and Fit the Preprocessor

Important: the preprocessor is **fitted only on `X_train`**.

The fitted training preprocessor is then used to transform validation and test features.  
This prevents information leakage from future data.


In [ ]:
preprocessor = build_preprocessor(X_train)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

# Prepare test features with the same fitted transformer,
# but do not use y_test for baseline model selection.
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed.")
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)


## 6. Create the Baseline Random Forest

No hyperparameter search is performed here.

`random_state=42` is used so the experiment is reproducible.


In [ ]:
rf_baseline = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
)

rf_baseline


## 7. Train the Model


In [ ]:
rf_baseline.fit(X_train_processed, y_train)

print("Random Forest baseline training completed.")


## 8. Predict the Validation Set

The validation set is used to evaluate this baseline model.  
The test labels remain untouched for later final-model evaluation.


In [ ]:
y_val_pred = rf_baseline.predict(X_val_processed)

print("Validation predictions completed.")


## 9. Baseline Evaluation Metrics

Because the target classes are imbalanced, we report both:

- Accuracy
- Macro F1 Score

Macro F1 gives equal importance to each class.


In [ ]:
accuracy = accuracy_score(y_val, y_val_pred)
macro_f1 = f1_score(y_val, y_val_pred, average="macro")

print("RANDOM FOREST BASELINE - VALIDATION RESULTS")
print("=" * 55)
print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")


## 10. Classification Report

This shows precision, recall, and F1-score for each target class.


In [ ]:
print(
    classification_report(
        y_val,
        y_val_pred,
        zero_division=0,
    )
)


## 11. Confusion Matrix

The confusion matrix shows which risk categories the model predicts correctly and which categories are confused with each other.


In [ ]:
labels = sorted(pd.Series(y_val).dropna().unique())

cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=labels,
)

fig, ax = plt.subplots(figsize=(10, 8))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels,
)

display.plot(
    ax=ax,
    xticks_rotation=45,
    values_format="d",
    colorbar=False,
)

ax.set_title("Random Forest Baseline - Validation Confusion Matrix")
plt.tight_layout()

# Save the confusion matrix image for the report/presentation
pictures_dir = project_root / "reports" / "pictures"
pictures_dir.mkdir(parents=True, exist_ok=True)

picture_path = pictures_dir / "random_forest_baseline_confusion_matrix.png"
plt.savefig(picture_path, dpi=300, bbox_inches="tight")

print("Confusion matrix saved to:", picture_path)

plt.show()


## 12. Save Overall Baseline Metrics

Save the main validation metrics so they can later be compared with Gradient Boosting and the other group models.


In [ ]:
results_dir = project_root / "reports" / "model_results"
pictures_dir = project_root / "reports" / "pictures"

results_dir.mkdir(parents=True, exist_ok=True)
pictures_dir.mkdir(parents=True, exist_ok=True)

baseline_results = pd.DataFrame(
    [{
        "model": "Random Forest Baseline",
        "validation_accuracy": accuracy,
        "validation_macro_f1": macro_f1,
        "hyperparameter_tuning": False,
        "model_optimization": False,
    }]
)

baseline_results_path = results_dir / "random_forest_baseline_results.csv"
baseline_results.to_csv(baseline_results_path, index=False)

print("Saved:", baseline_results_path)
baseline_results


## 13. Save the Classification Report

This preserves class-wise precision, recall, F1-score, and support as a CSV file for the report and later model comparison.


In [ ]:
classification_report_dict = classification_report(
    y_val,
    y_val_pred,
    output_dict=True,
    zero_division=0,
)

classification_report_df = pd.DataFrame(classification_report_dict).transpose()

classification_report_path = (
    results_dir / "random_forest_classification_report.csv"
)
classification_report_df.to_csv(classification_report_path)

print("Saved:", classification_report_path)
classification_report_df


## 14. Save Accuracy vs Macro F1 Diagram

The difference between overall accuracy and Macro F1 is especially useful when discussing class imbalance.


In [ ]:
overall_metrics = pd.Series(
    {
        "Accuracy": accuracy,
        "Macro F1": macro_f1,
    }
)

fig, ax = plt.subplots(figsize=(7, 5))
overall_metrics.plot(kind="bar", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Random Forest Baseline - Validation Metrics")
ax.tick_params(axis="x", rotation=0)

for i, value in enumerate(overall_metrics.values):
    ax.text(i, min(value + 0.025, 0.97), f"{value:.3f}", ha="center")

plt.tight_layout()

overall_metrics_picture = (
    pictures_dir / "random_forest_baseline_metrics.png"
)
plt.savefig(overall_metrics_picture, dpi=300, bbox_inches="tight")
print("Saved:", overall_metrics_picture)
plt.show()


## 15. Save Class-wise Precision, Recall and F1 Diagram

This diagram makes it easier to see whether the model performs differently across PM2.5 risk classes.


In [ ]:
class_names = [
    label for label in labels
    if label in classification_report_df.index
]

class_metrics = classification_report_df.loc[
    class_names,
    ["precision", "recall", "f1-score"]
]

ax = class_metrics.plot(kind="bar", figsize=(12, 6))
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_xlabel("PM2.5 Risk Category")
ax.set_title("Random Forest Baseline - Class-wise Validation Metrics")
ax.tick_params(axis="x", rotation=45)
plt.legend(title="Metric")
plt.tight_layout()

class_metrics_picture = (
    pictures_dir / "random_forest_class_metrics.png"
)
plt.savefig(class_metrics_picture, dpi=300, bbox_inches="tight")
print("Saved:", class_metrics_picture)
plt.show()


## 16. Feature Importance

Random Forest provides feature-importance scores. These scores help identify which transformed input features contributed most to the model's decisions.

Feature importance is used here for **interpretation of the baseline model**, not as a feature-selection optimization step.


In [ ]:
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    feature_names = np.array(
        [f"feature_{i}" for i in range(X_train_processed.shape[1])]
    )

feature_importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": rf_baseline.feature_importances_,
    }
).sort_values("importance", ascending=False)

feature_importance_path = (
    results_dir / "random_forest_feature_importance.csv"
)
feature_importance_df.to_csv(feature_importance_path, index=False)

print("Saved:", feature_importance_path)
feature_importance_df.head(20)


## 17. Save Top Feature Importance Diagram

For readability, the diagram displays the 15 most important transformed features.


In [ ]:
top_features = feature_importance_df.head(15).sort_values(
    "importance",
    ascending=True,
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_features["feature"], top_features["importance"])
ax.set_xlabel("Feature Importance")
ax.set_ylabel("Feature")
ax.set_title("Random Forest Baseline - Top 15 Feature Importances")
plt.tight_layout()

feature_importance_picture = (
    pictures_dir / "random_forest_feature_importance.png"
)
plt.savefig(feature_importance_picture, dpi=300, bbox_inches="tight")
print("Saved:", feature_importance_picture)
plt.show()


## 18. Output Check

The baseline experiment should now produce:

**CSV evidence**
- `random_forest_baseline_results.csv`
- `random_forest_classification_report.csv`
- `random_forest_feature_importance.csv`

**Figures**
- `random_forest_baseline_confusion_matrix.png`
- `random_forest_baseline_metrics.png`
- `random_forest_class_metrics.png`
- `random_forest_feature_importance.png`


In [ ]:
expected_outputs = [
    results_dir / "random_forest_baseline_results.csv",
    results_dir / "random_forest_classification_report.csv",
    results_dir / "random_forest_feature_importance.csv",
    pictures_dir / "random_forest_baseline_confusion_matrix.png",
    pictures_dir / "random_forest_baseline_metrics.png",
    pictures_dir / "random_forest_class_metrics.png",
    pictures_dir / "random_forest_feature_importance.png",
]

print("RANDOM FOREST BASELINE OUTPUT CHECK")
print("=" * 55)

for output_path in expected_outputs:
    status = "OK" if output_path.exists() else "MISSING"
    print(f"{status:7} | {output_path}")


## Final Baseline Model Summary

The Random Forest baseline was trained on the chronological training set and evaluated on the validation set.

The preprocessing transformer was fitted only on training features and then applied to validation data to avoid information leakage.

The notebook records overall performance, class-wise performance, prediction errors through a confusion matrix, and Random Forest feature importance.

No hyperparameter tuning or model optimization is performed here. These results form the **baseline evidence** that can later be compared with Gradient Boosting, optimized models, and the other algorithms developed by the group.

The chronological test set is kept separate from baseline model selection and should be reserved for final-model evaluation.
